# UNSW-NB15: cited-paper reproduction and research extension

This notebook is a runnable, step-by-step Colab workflow for (1) reproducing the cited paper's binary classifier comparison on the official UNSW-NB15 partition, (2) reproducing the paper's reported Random Forest binary reference and testing the paper's proposed cost-sensitive GBDT extension, and (3) comparing a ten-class `attack_cat` Random Forest reference with a LightGBM + SMOTENC extension. It writes intermediate and final results to Google Drive.

**Official data files:** `UNSW_NB15_training-set.csv` (175,341 rows) and `UNSW_NB15_testing-set.csv` (82,332 rows), from the UNSW-NB15 dataset distribution. Keep this fixed official train/test split; do not randomly split or rebalance the test set. Expected target columns are `label` (binary) and `attack_cat` (ten classes including `Normal`). For multi-class, features exclude `id`, `label`, and `attack_cat`; `proto`, `service`, and `state` are categorical.

**Resampling policy:** the binary paper reproduction does not use SMOTE. In the multi-class extension, SMOTENC is fitted only on each training fold during CV and once on the complete training partition for each final model fit. No validation/test observations are resampled. Preprocessors are fit within training folds or on the training partition only.

**Runtime:** the seven binary models and repeated experiments can take a long time, especially KNN/SVM and cross-validation. Run sections independently, save outputs to Drive, and use the switches below. A GPU is not guaranteed to accelerate these tree models. Notebook-generated values are new measurements; they can differ slightly from the paper due to library versions and because the cited paper omits some implementation details. Do not present planned values as measured results.

Citation/context: Omarov et al. (2024), the cited source paper in the project; this notebook records implementation choices required to reproduce its reported comparison. The supplied official split files should be obtained from the dataset's official UNSW-NB15 distribution and uploaded to Drive by the researcher.

## 1. Install dependencies and record versions
Run once per new Colab runtime. scikit-learn 1.9.1 and LightGBM 4.7.0 match the environment recorded in the paper; imbalanced-learn is recorded below. Versions are printed to make the run auditable.

In [ ]:
%pip -q install scikit-learn==1.9.1 lightgbm==4.7.0 imbalanced-learn==0.14.2 matplotlib scipy
import sklearn, imblearn, lightgbm, pandas, numpy
print('scikit-learn', sklearn.__version__)
print('imbalanced-learn', imblearn.__version__)
print('LightGBM', lightgbm.__version__)
print('pandas', pandas.__version__, '| numpy', numpy.__version__)

## 2. Connect Drive and select the official files
The notebook reads the two official CSVs from `MyDrive/UNSW_NB15/Dataset/`. Drive is mounted only to read those input files. Generated models, reports, and figures are written to the Colab runtime under `/content/UNSW_NB15_colab_results/`; they are not written to Drive. The final section can package and download them before the runtime ends. The assertions below stop the notebook if the expected partition sizes or columns do not match. Do not substitute a Kaggle re-split for these files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil
DATA_DIR = Path('/content/drive/MyDrive/UNSW_NB15/Dataset')
TRAIN_PATH = DATA_DIR / 'UNSW_NB15_training-set.csv'
TEST_PATH = DATA_DIR / 'UNSW_NB15_testing-set.csv'
OUT_DIR = Path('/content/UNSW_NB15_colab_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert TRAIN_PATH.exists() and TEST_PATH.exists(), f'Place both official CSVs in {DATA_DIR}'
print(TRAIN_PATH, TRAIN_PATH.stat().st_size, 'bytes')
print(TEST_PATH, TEST_PATH.stat().st_size, 'bytes')
print('Local Colab output directory:', OUT_DIR)

## 3. Load data, validate schema, and document the split
Binary target: `label` (0 Normal, 1 attack). Multi-class target: `attack_cat`. The official test partition is preserved as provided. The data checks report class counts and feature overlap for transparency; they do not alter the data.

In [ ]:
import numpy as np, pandas as pd, json, time, warnings, itertools, os
from IPython.display import display
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
assert len(train_df) == 175341 and len(test_df) == 82332, (len(train_df), len(test_df))
CAT = ['proto', 'service', 'state']
assert set(CAT).issubset(train_df.columns) and set(CAT).issubset(test_df.columns)
assert {'id','label','attack_cat'}.issubset(train_df.columns)
assert train_df.columns.equals(test_df.columns)
print('Split rows:', len(train_df), 'train /', len(test_df), 'test')
print('Binary train counts:'); display(train_df.label.value_counts().sort_index().rename('count').to_frame())
print('Binary test counts:'); display(test_df.label.value_counts().sort_index().rename('count').to_frame())
print('Attack-category training counts:'); display(train_df.attack_cat.value_counts().to_frame('count'))
print('Attack-category test counts:'); display(test_df.attack_cat.value_counts().to_frame('count'))
print('Columns:', len(train_df.columns), '| categorical:', CAT)
# Duplicate/overlap audit used in the paper; labels are inspected for conflicting identical features.
_aX=train_df.drop(columns=['id','attack_cat','label']); _bX=test_df.drop(columns=['id','attack_cat','label'])
_ha=pd.util.hash_pandas_object(_aX,index=False).to_numpy(); _hb=pd.util.hash_pandas_object(_bX,index=False).to_numpy()
_hall=np.concatenate([_ha,_hb]); _labels=np.concatenate([train_df.label.to_numpy(),test_df.label.to_numpy()])
_conflicts=pd.DataFrame({'hash':_hall,'label':_labels}).groupby('hash').label.nunique()
print('Duplicate audit:',{'unique_train':len(np.unique(_ha)),'unique_test':len(np.unique(_hb)),
 'test_rows_also_in_train':int(np.isin(_hb,_ha).sum()),'feature_vectors_with_conflicting_binary_labels':int((_conflicts>1).sum()),
 'train_attack_prior':float(train_df.label.mean()),'test_attack_prior':float(test_df.label.mean())})


## 4. Shared preprocessing and metrics
All feature transformations are fitted inside the pipeline on the corresponding training data. One-hot encoding uses `handle_unknown='ignore'` for the binary models. For attack category, the feature set excludes the binary label to prevent target leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, roc_auc_score,
                             confusion_matrix, matthews_corrcoef, classification_report)
def onehot():
    # sparse_output introduced in newer sklearn; compatibility fallback for older Colab runtimes
    try: return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError: return OneHotEncoder(handle_unknown='ignore', sparse=False)
def binary_xy(df):
    X = df.drop(columns=['id','attack_cat','label'])
    y = df['label'].astype(int).to_numpy()
    return X, y
Xtr, ytr = binary_xy(train_df); Xte, yte = binary_xy(test_df)
NUM = [c for c in Xtr.columns if c not in CAT]
def binary_preprocessor(scale=True):
    numeric_pipe = Pipeline([('scale', StandardScaler())]) if scale else 'passthrough'
    cat_pipe = onehot()
    return ColumnTransformer([('num', numeric_pipe, NUM), ('cat', cat_pipe, CAT)], remainder='drop')
def binary_metrics(y, pred, score=None):
    pm, rm, fm, _ = precision_recall_fscore_support(y,pred,average='macro',zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(y,pred,average='weighted',zero_division=0)
    pa, ra, fa, _ = precision_recall_fscore_support(y,pred,average='binary',pos_label=1,zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
    d={'accuracy':accuracy_score(y,pred),'macro_precision':pm,'macro_recall':rm,'macro_f1':fm,
       'weighted_precision':pw,'weighted_recall':rw,'weighted_f1':fw,
       'attack_precision':pa,'attack_recall':ra,'attack_f1':fa,
       'FAR':fp/(fp+tn),'FNR':fn/(fn+tp),'MCC':matthews_corrcoef(y,pred),
       'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp)}
    if score is not None: d['AUC']=roc_auc_score(y,score)
    return d
SEEDS = [0,1,2,3,4]

## 5. Reproduce the cited paper's binary model comparison
This runs KNN, Gaussian Naive Bayes, Decision Tree, Logistic Regression, AdaBoost, Random Forest, and RBF-SVM on the official split. The source paper does not fully specify every preprocessing choice or all hyperparameters; this reproduction uses scikit-learn defaults, with a train-only StandardScaler + one-hot preprocessing pipeline, and documents that operational choice. DT/AdaBoost/RF are repeated for five seeds. The paper's SVM comparison uses a stratified 40,000-row subset of the training partition (to limit its cost); the cell follows that convention.

Set `RUN_BINARY_BASELINES=False` to skip if you only need the extension. Results are checkpointed after each model.

In [ ]:
RUN_BINARY_BASELINES = True
if RUN_BINARY_BASELINES:
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.naive_bayes import GaussianNB
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.svm import SVC
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
    from sklearn.model_selection import train_test_split
    baseline_specs = {
      'KNN': (lambda s: KNeighborsClassifier(n_jobs=2), [0]),
      'Naive Bayes': (lambda s: GaussianNB(), [0]),
      'Decision Tree': (lambda s: DecisionTreeClassifier(random_state=s), SEEDS),
      'Logistic Regression': (lambda s: LogisticRegression(max_iter=1000), [0]),
      'AdaBoost': (lambda s: AdaBoostClassifier(random_state=s), SEEDS),
      'Random Forest': (lambda s: RandomForestClassifier(n_estimators=100,n_jobs=2,random_state=s), SEEDS),
    }
    binary_baselines = {}
    for name,(make_model,seeds) in baseline_specs.items():
        binary_baselines[name]=[]
        for seed in seeds:
            pipe=Pipeline([('prep',binary_preprocessor(scale=True)),('model',make_model(seed))])
            t0=time.time(); pipe.fit(Xtr,ytr); fit_s=time.time()-t0
            pred=pipe.predict(Xte); score=pipe.predict_proba(Xte)[:,1] if hasattr(pipe,'predict_proba') else pipe.decision_function(Xte)
            row=binary_metrics(yte,pred,score); row.update(seed=seed,fit_seconds=fit_s)
            binary_baselines[name].append(row); print(name,seed,'accuracy',round(row['accuracy'],4),'macro-F1',round(row['macro_f1'],4),'fit(s)',round(fit_s,1))
        with open(OUT_DIR/'binary_baselines_partial.json','w') as f: json.dump(binary_baselines,f,indent=2)
    # Reproduce the paper's reduced SVM training subset; test remains the complete official test file.
    Xsvm, _, ysvm, _ = train_test_split(Xtr,ytr,train_size=40000,random_state=0,stratify=ytr)
    svm_pipe=Pipeline([('prep',binary_preprocessor(scale=True)),('model',SVC(kernel='rbf'))])
    t0=time.time(); svm_pipe.fit(Xsvm,ysvm); svm_fit_s=time.time()-t0
    svm_pred=svm_pipe.predict(Xte); svm_score=svm_pipe.decision_function(Xte)
    svm_row=binary_metrics(yte,svm_pred,svm_score); svm_row.update(seed=0,train_rows=40000,fit_seconds=svm_fit_s)
    binary_baselines['SVM (40k stratified train subset)']=[svm_row]
    with open(OUT_DIR/'binary_baselines.json','w') as f: json.dump(binary_baselines,f,indent=2)
    display(pd.DataFrame([{ 'model':k, **{m:np.mean([r[m] for r in v]) for m in ['accuracy','macro_f1','FAR','FNR','AUC']} } for k,v in binary_baselines.items()]).set_index('model').style.format('{:.4f}'))

## 6. Reproduce the binary RF reference and evaluate the proposed cost-sensitive GBDT
The extension uses duplicate-aware `StratifiedGroupKFold` on hashes of feature rows. All threshold selection and hyperparameter selection use training-partition out-of-fold probabilities only. The operating threshold maximizes Youden's J (`TPR-FPR`) on those OOF probabilities. The official test split is used once for final evaluation.

The proposed binary model is LightGBM with class-balanced weights, 300 estimators, learning rate 0.05, 0.8 row/column subsampling, and a grouped-CV grid over `num_leaves={15,31,63}` and `min_child_samples={20,100}`. To reduce runtime, set the seed list smaller for a smoke run, but report the final settings and number of seeds actually run.

This section can be lengthy. It saves after each seed so interrupted Colab sessions retain completed test scores.

In [ ]:
RUN_BINARY_EXTENSION = True
if RUN_BINARY_EXTENSION:
    import lightgbm as lgb
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.metrics import roc_curve
    # Keep categories nominal for LightGBM. Category vocabulary is learned from training partition only.
    def lgb_categories(X):
        out=X.copy()
        for c in CAT:
            levels=pd.Index(out[c].astype(str).value_counts().index)
            out[c]=pd.Categorical(out[c].astype(str),categories=levels)
        return out
    XA=lgb_categories(Xtr); XB=Xte.copy()
    for c in CAT: XB[c]=pd.Categorical(XB[c].astype(str),categories=XA[c].cat.categories)
    hashes=pd.util.hash_pandas_object(Xtr,index=False).to_numpy()
    group_cv=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=0)
    folds=list(group_cv.split(XA,ytr,groups=hashes))
    BASE={'n_estimators':300,'learning_rate':0.05,'subsample':0.8,'subsample_freq':1,
          'colsample_bytree':0.8,'n_jobs':2,'verbosity':-1}
    def make_gbdt(seed=0,balanced=True,**kw):
        return lgb.LGBMClassifier(**{**BASE,**kw},random_state=seed,
             class_weight='balanced' if balanced else None)
    def oof_gbdt(columns,balanced=True,**kw):
        oof=np.zeros(len(ytr))
        for fold,(fit_i,val_i) in enumerate(folds,1):
            model=make_gbdt(0,balanced,**kw)
            model.fit(XA.iloc[fit_i][columns],ytr[fit_i],categorical_feature=CAT)
            oof[val_i]=model.predict_proba(XA.iloc[val_i][columns])[:,1]
            print('OOF fold',fold,'complete')
        return oof
    def youden_threshold(y,prob):
        fpr,tpr,thresholds=roc_curve(y,prob)
        return float(thresholds[np.argmax(tpr-fpr)])
    grid=[]; best=None
    for leaves,min_child in itertools.product([15,31,63],[20,100]):
        prob=oof_gbdt(list(XA.columns),True,num_leaves=leaves,min_child_samples=min_child)
        auc=roc_auc_score(ytr,prob); grid.append({'num_leaves':leaves,'min_child_samples':min_child,'oof_auc':auc})
        print(grid[-1])
        if best is None or auc>best[0]: best=(auc,{'num_leaves':leaves,'min_child_samples':min_child},prob)
    hp=best[1]; oof_bal=best[2]; threshold_bal=youden_threshold(ytr,oof_bal)
    oof_plain=oof_gbdt(list(XA.columns),False,**hp); threshold_plain=youden_threshold(ytr,oof_plain)
    # Grouped OOF RF threshold for the paper's balanced RF ablation.
    def rf_pipe(seed,balanced=False):
        return Pipeline([('prep',binary_preprocessor(scale=True)),('model',RandomForestClassifier(
            n_estimators=100,n_jobs=2,random_state=seed,class_weight='balanced' if balanced else None))])
    oof_rf=np.zeros(len(ytr))
    for fold,(fit_i,val_i) in enumerate(folds,1):
        p=rf_pipe(0,True).fit(Xtr.iloc[fit_i],ytr[fit_i]).predict_proba(Xtr.iloc[val_i])[:,1]
        oof_rf[val_i]=p; print('RF OOF fold',fold,'complete')
    threshold_rf=youden_threshold(ytr,oof_rf)
    variants={k:[] for k in ['RF baseline (0.5)','RF + balanced','RF + balanced + J threshold',
       'GBDT (0.5)','GBDT + J threshold','GBDT + balanced','GBDT + balanced + J threshold (proposed)']}
    seed0_scores={}
    EXTENSION_SEEDS=[0,1,2,3,4]
    for seed in EXTENSION_SEEDS:
        p=rf_pipe(seed,False).fit(Xtr,ytr).predict_proba(Xte)[:,1]
        variants['RF baseline (0.5)'].append(binary_metrics(yte,p>=0.5,p))
        if seed==0: seed0_scores['rf']=p
        p=rf_pipe(seed,True).fit(Xtr,ytr).predict_proba(Xte)[:,1]
        variants['RF + balanced'].append(binary_metrics(yte,p>=0.5,p))
        variants['RF + balanced + J threshold'].append(binary_metrics(yte,p>=threshold_rf,p))
        plain=make_gbdt(seed,False,**hp).fit(XA,ytr,categorical_feature=CAT).predict_proba(XB)[:,1]
        variants['GBDT (0.5)'].append(binary_metrics(yte,plain>=0.5,plain))
        variants['GBDT + J threshold'].append(binary_metrics(yte,plain>=threshold_plain,plain))
        bal_model=make_gbdt(seed,True,**hp).fit(XA,ytr,categorical_feature=CAT)
        bal=bal_model.predict_proba(XB)[:,1]
        variants['GBDT + balanced'].append(binary_metrics(yte,bal>=0.5,bal))
        variants['GBDT + balanced + J threshold (proposed)'].append(binary_metrics(yte,bal>=threshold_bal,bal))
        if seed==0: seed0_scores['proposed']=bal; model0=bal_model
        with open(OUT_DIR/'binary_extension_partial.json','w') as f:
            json.dump({'grid':grid,'selected_hyperparameters':hp,'thresholds':{'RF':threshold_rf,'GBDT_plain':threshold_plain,'GBDT_balanced':threshold_bal},'variants':variants},f,indent=2)
        print('seed',seed,'completed')
    rows=[]
    for name,runs in variants.items():
        row={'model':name}
        for m in ['accuracy','macro_f1','FAR','FNR','AUC','MCC']:
            vals=[r[m] for r in runs]; row[m+'_mean']=float(np.mean(vals)); row[m+'_sd']=float(np.std(vals,ddof=1)) if len(vals)>1 else 0.0
        rows.append(row)
    display(pd.DataFrame(rows).set_index('model').style.format('{:.4f}'))
    with open(OUT_DIR/'binary_extension.json','w') as f:
        json.dump({'grid':grid,'selected_hyperparameters':hp,'thresholds':{'RF':threshold_rf,'GBDT_plain':threshold_plain,'GBDT_balanced':threshold_bal},'variants':variants},f,indent=2)
    np.savez(OUT_DIR/'binary_seed0_scores.npz',y=yte,rf=seed0_scores['rf'],proposed=seed0_scores['proposed'],threshold=threshold_bal)

## 7. Paper diagnostics: significance, per-category analysis, and ROC figures
This uses the seed-0 test predictions from the fitted binary models. It reproduces the paper's exploratory row-level McNemar/bootstrap analysis and category-wise detector rates, then exports ROC and per-category figures as PDF and PNG. The test set is used only for final reporting/visualization; it is not used for fitting, selecting hyperparameters, or choosing the operating threshold.

In [ ]:
from scipy.stats import chi2
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
rf_prob=seed0_scores['rf']; prop_prob=seed0_scores['proposed']
rf_pred=rf_prob >= 0.5; prop_pred=prop_prob >= threshold_bal
rf_correct=(rf_pred == yte); prop_correct=(prop_pred == yte)
n01=int((~rf_correct & prop_correct).sum()); n10=int((rf_correct & ~prop_correct).sum())
mcnemar_chi2=(abs(n01-n10)-1)**2/(n01+n10)
_rng=np.random.default_rng(0); _gains=[]
for _ in range(2000):
    _idx=_rng.integers(0,len(yte),len(yte)); _gains.append(100*(prop_correct[_idx].mean()-rf_correct[_idx].mean()))
print({'RF_wrong/proposed_right':n01,'RF_right/proposed_wrong':n10,'McNemar_chi2':mcnemar_chi2,
       'p_value':float(chi2.sf(mcnemar_chi2,1)),
       'accuracy_gain_CI95_pp':np.percentile(_gains,[2.5,97.5]).tolist()})
cat_test=test_df['attack_cat'].astype(str).to_numpy(); cat_results={}
for cat in sorted(set(cat_test)):
    mask=cat_test==cat; target=0 if cat=='Normal' else 1
    cat_results[cat]={'n':int(mask.sum()),
      'RF_correct_%':100*float((rf_pred[mask]==target).mean()),
      'Proposed_correct_%':100*float((prop_pred[mask]==target).mean())}
cat_frame=pd.DataFrame(cat_results).T; display(cat_frame)
with open(OUT_DIR/'binary_diagnostics.json','w') as f:
    json.dump({'mcnemar':{'n01':n01,'n10':n10,'chi2':float(mcnemar_chi2),
       'p_value':float(chi2.sf(mcnemar_chi2,1)),
       'accuracy_gain_ci95_pp':np.percentile(_gains,[2.5,97.5]).tolist()},
       'per_category':cat_results},f,indent=2)
# ROC panel; operating points correspond to the paper's RF default and proposed J threshold.
plt.rcParams.update({'font.family':'serif','font.size':8,'axes.spines.top':False,'axes.spines.right':False})
fig,ax=plt.subplots(figsize=(3.4,2.5))
for score,color,label in [(rf_prob,'#2a78d6','RF baseline'),(prop_prob,'#eb6834','Proposed GBDT')]:
    fpr,tpr,_=roc_curve(yte,score); ax.plot(100*fpr,100*tpr,color=color,lw=1.4,label=label)
for score,pred,color,marker,label in [(rf_prob,rf_pred,'#2a78d6','o','RF, default 0.5'),
    (prop_prob,prop_pred,'#eb6834','s','Proposed, J threshold')]:
    fpr=100*np.mean(pred[yte==0]); tpr=100*np.mean(pred[yte==1])
    ax.plot(fpr,tpr,marker,color=color,ms=6,mec='white',mew=.8,label=f'{label}: FAR {fpr:.1f}%, DR {tpr:.1f}%')
ax.set_xlim(0,50); ax.set_ylim(70,100.5); ax.set_xlabel('False alarm rate (%)'); ax.set_ylabel('Detection rate (%)')
ax.grid(color='#dddddd',lw=.4); ax.legend(frameon=False,fontsize=6); fig.tight_layout(pad=.3)
fig.savefig(OUT_DIR/'roc.pdf'); fig.savefig(OUT_DIR/'roc.png',dpi=200); plt.show(); plt.close(fig)
# Category plot in paper order: Normal first, then attacks ordered by test support.
ordered=['Normal']+sorted([c for c in cat_results if c!='Normal'],key=lambda c:-cat_results[c]['n'])
ypos=np.arange(len(ordered))[::-1]; h=.36
fig,ax=plt.subplots(figsize=(3.4,3.0))
ax.barh(ypos+h/2+.02,[cat_results[c]['RF_correct_%'] for c in ordered],h,color='#2a78d6',label='RF baseline')
ax.barh(ypos-h/2-.02,[cat_results[c]['Proposed_correct_%'] for c in ordered],h,color='#eb6834',label='Proposed GBDT')
ax.set_yticks(ypos); ax.set_yticklabels([f"{c} ({cat_results[c]['n']:,})" for c in ordered])
ax.set_xlim(0,112); ax.set_xticks([0,25,50,75,100]); ax.set_xlabel('Correctly classified test flows (%)')
ax.grid(axis='x',color='#dddddd',lw=.4); ax.set_axisbelow(True); ax.legend(frameon=False,fontsize=7,loc='lower center',bbox_to_anchor=(.4,1),ncol=2)
fig.tight_layout(pad=.3); fig.savefig(OUT_DIR/'percat.pdf'); fig.savefig(OUT_DIR/'percat.png',dpi=200); plt.show(); plt.close(fig)
print('Saved diagnostic JSON and ROC/per-category figures to',OUT_DIR)

## 8. Paper compact-feature and runtime analysis (optional, extra training)
The paper reports feature-gain ranking, five compact variants, and single-thread runtime/model-size comparisons. This cell recreates those experiments from the selected binary model and grouped folds. It is intentionally switchable because it adds 25 grouped-CV model fits and several final fits. Keep `RUN_COMPACT_COST=True` to reproduce the full paper analysis.

In [ ]:
RUN_COMPACT_COST = True
if RUN_BINARY_EXTENSION and RUN_COMPACT_COST:
    import pickle
    # Gain ranking is taken from the seed-0, full-feature proposed GBDT fit.
    gain=pd.Series(model0.booster_.feature_importance('gain'),index=list(XA.columns)).sort_values(ascending=False)
    importance=(100*gain/gain.sum()).round(3)
    compact_results={}; compact_models={}
    for k in [5,10,15,20,30]:
        cols=list(gain.index[:k])
        oof_compact=oof_gbdt(cols,True,**hp)
        tau=youden_threshold(ytr,oof_compact)
        compact_model=make_gbdt(0,True,**hp).fit(XA[cols],ytr)
        p=compact_model.predict_proba(XB[cols])[:,1]
        compact_results[k]={'features':cols,'threshold':tau,
          'cv_auc_%':100*roc_auc_score(ytr,oof_compact),
          **binary_metrics(yte,p>=tau,p)}
        compact_models[k]=compact_model
        print('top',k,'features: acc',round(100*compact_results[k]['accuracy'],2),
              'macro-F1',round(100*compact_results[k]['macro_f1'],2),
              'FAR',round(compact_results[k]['FAR'],2),'FNR',round(compact_results[k]['FNR'],2))
    print('Top gain features (%):'); display(importance.head(15).rename('gain_share_percent').to_frame())
    with open(OUT_DIR/'binary_compact_results.json','w') as f: json.dump(compact_results,f,indent=2)
    # Match the paper's separate one-thread timing protocol; times are runtime-dependent.
    timing={}
    def benchmark(name,model,Xa,Xb):
        started=time.time(); model.fit(Xa,ytr); fit_sec=time.time()-started
        pred_times=[]
        for _ in range(3):
            started=time.time(); model.predict_proba(Xb); pred_times.append(time.time()-started)
        timing[name]={'fit_seconds':fit_sec,
          'microseconds_per_test_flow':1e6*min(pred_times)/len(Xb),
          'serialized_model_MB':len(pickle.dumps(model))/2**20}
        print(name,timing[name])
    prep_timing=ColumnTransformer([('num',StandardScaler(),NUM),('cat',onehot(),CAT)])
    rf_timing=Pipeline([('prep',prep_timing),('clf',RandomForestClassifier(n_estimators=100,n_jobs=1,random_state=0))])
    benchmark('RF',rf_timing,Xtr,Xte)
    lgb_base={'n_estimators':300,'learning_rate':0.05,'subsample':0.8,'subsample_freq':1,
      'colsample_bytree':0.8,'verbosity':-1,'n_jobs':1,'random_state':0,'class_weight':'balanced',**hp}
    for name,cols in [('GBDT-42',list(XA.columns)),('GBDT-10',compact_results[10]['features']),('GBDT-20',compact_results[20]['features'])]:
        timing_model=lgb.LGBMClassifier(**lgb_base)
        benchmark(name,timing_model,XA[cols],XB[cols])
    with open(OUT_DIR/'binary_timing.json','w') as f: json.dump(timing,f,indent=2)
    with open(OUT_DIR/'binary_feature_importance.json','w') as f: json.dump(importance.to_dict(),f,indent=2)
    print('Compact variants and timing artifacts saved in',OUT_DIR)

## 9. Ten-class `attack_cat`: baseline and SMOTENC extension
Classes are discovered from training data and checked to include all ten values including `Normal`. Numeric columns use median imputation. Categorical columns use most-frequent imputation and ordinal encoding fitted within the fold; SMOTENC receives the three categorical column positions and preserves their categorical status while generating samples. The chosen policy oversamples any class below 5,000 training rows up to that floor.

Cross-validation uses `StratifiedKFold(5, shuffle=True, random_state=0)` with class stratification. Duplicate feature vectors are not grouped for this ten-class experiment, as described in the paper. For every fold, the validation fold is never resampled. For final evaluation, SMOTENC is applied once to all training rows only; the original 82,332-row test set is transformed and predicted without resampling. Both arms are trained/evaluated on the same official split. RF is the reproduced multi-class reference, with five seeds.

Reported metrics include each class's precision, recall, F1, support, macro-F1, and accuracy. SMOTENC results are generated here; no class improvement should be claimed until this cell finishes and the saved results are inspected.

In [ ]:
RUN_MULTICLASS = True
if RUN_MULTICLASS:
    from imblearn.over_sampling import SMOTENC
    from imblearn.pipeline import Pipeline as ImbPipeline
    from sklearn.model_selection import StratifiedKFold
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import precision_recall_fscore_support
    TARGET='attack_cat'; EXCLUDE={'id','label',TARGET}
    mc_features=[c for c in train_df.columns if c not in EXCLUDE]
    Xmc=train_df[mc_features].copy(); ymc=train_df[TARGET].astype(str).str.strip().reset_index(drop=True)
    Xmc_test=test_df[mc_features].copy(); ymc_test=test_df[TARGET].astype(str).str.strip().reset_index(drop=True)
    classes=sorted(ymc.unique()); assert len(classes)==10 and 'Normal' in classes, classes
    assert set(ymc_test.unique()).issubset(set(classes))
    numeric_mc=[c for c in mc_features if c not in CAT]
    labels=np.arange(len(classes)); SEEDS=[0,1,2,3,4]
    def mc_metrics(y_true,y_pred):
        p,r,f,s=precision_recall_fscore_support(y_true,y_pred,labels=classes,zero_division=0)
        return {'accuracy':float(accuracy_score(y_true,y_pred)),
          'macro_precision':float(p.mean()),'macro_recall':float(r.mean()),'macro_f1':float(f.mean()),
          'per_class':{c:{'precision':float(p[i]),'recall':float(r[i]),'f1':float(f[i]),'support':int(s[i])} for i,c in enumerate(classes)}}
    def mc_preprocessor():
        return ColumnTransformer([
          ('numeric',SimpleImputer(strategy='median'),numeric_mc),
          ('categorical',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),
             ('ordinal',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1))]),CAT)],
          remainder='drop',verbose_feature_names_out=False)
    def mc_smote_pipeline(seed,y_fit):
        counts=y_fit.value_counts()
        strategy={label:5000 for label,count in counts.items() if count<5000}
        sampler=SMOTENC(categorical_features=list(range(len(numeric_mc),len(numeric_mc)+len(CAT))),
             sampling_strategy=strategy if strategy else 'auto',random_state=seed)
        model=lgb.LGBMClassifier(objective='multiclass',n_estimators=300,learning_rate=0.05,
             num_leaves=31,colsample_bytree=0.8,subsample=0.8,subsample_freq=1,
             n_jobs=2,random_state=seed,verbosity=-1)
        return ImbPipeline([('preprocess',mc_preprocessor()),('smotenc',sampler),('classifier',model)])
    def mc_rf_pipeline(seed):
        prep=ColumnTransformer([('numeric',StandardScaler(),numeric_mc),
          ('categorical',onehot(),CAT)])
        return Pipeline([('preprocess',prep),('classifier',RandomForestClassifier(n_estimators=100,n_jobs=2,random_state=seed))])
    # Five-fold class-stratified CV; duplicate rows are not grouped in this ten-class experiment.
    cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=0); oof=np.empty(len(ymc),dtype=object)
    for fold,(fit_i,val_i) in enumerate(cv.split(Xmc,ymc),1):
        pipe=mc_smote_pipeline(0,ymc.iloc[fit_i])
        pipe.fit(Xmc.iloc[fit_i],ymc.iloc[fit_i],classifier__categorical_feature=list(range(len(numeric_mc),len(numeric_mc)+len(CAT))))
        oof[val_i]=pipe.predict(Xmc.iloc[val_i])
        print('Stratified SMOTENC CV fold',fold,'complete; only fold-training rows are resampled')
    cv_result=mc_metrics(ymc,oof); print('SMOTENC OOF CV:',{k:v for k,v in cv_result.items() if k!='per_class'})
    # Final official-test evaluation. Each seed gets a fresh sampler fitted only to full training data.
    mc_runs={'RF baseline (reproduced)':[],'LightGBM + SMOTENC':[]}
    per_class_rows=[]
    for seed in SEEDS:
        rf=mc_rf_pipeline(seed).fit(Xmc,ymc); pred_rf=rf.predict(Xmc_test)
        res_rf=mc_metrics(ymc_test,pred_rf); res_rf['seed']=seed; mc_runs['RF baseline (reproduced)'].append(res_rf)
        sm=mc_smote_pipeline(seed,ymc).fit(Xmc,ymc,classifier__categorical_feature=list(range(len(numeric_mc),len(numeric_mc)+len(CAT)))) # training only
        pred_sm=sm.predict(Xmc_test); res_sm=mc_metrics(ymc_test,pred_sm); res_sm['seed']=seed
        mc_runs['LightGBM + SMOTENC'].append(res_sm)
        for model_name,res in [('RF baseline (reproduced)',res_rf),('LightGBM + SMOTENC',res_sm)]:
            for cls,vals in res['per_class'].items(): per_class_rows.append({'model':model_name,'seed':seed,'class':cls,**vals,'accuracy':res['accuracy'],'macro_f1':res['macro_f1']})
        with open(OUT_DIR/'multiclass_partial.json','w') as f:
            json.dump({'classes':classes,'cv_oof_smotenc':cv_result,'test_runs':mc_runs,
              'test_distribution':ymc_test.value_counts().to_dict(),'training_distribution':ymc.value_counts().to_dict(),
              'smote_floor':5000,'split':{'train':len(ymc),'test':len(ymc_test)}},f,indent=2)
        print('seed',seed,'RF',round(res_rf['accuracy'],4),round(res_rf['macro_f1'],4),
              '| SMOTENC',round(res_sm['accuracy'],4),round(res_sm['macro_f1'],4))
    summary=[]
    for model_name,runs in mc_runs.items():
        summary.append({'model':model_name,
          'accuracy_mean':np.mean([r['accuracy'] for r in runs]),'accuracy_sd':np.std([r['accuracy'] for r in runs],ddof=1),
          'macro_f1_mean':np.mean([r['macro_f1'] for r in runs]),'macro_f1_sd':np.std([r['macro_f1'] for r in runs],ddof=1)})
    display(pd.DataFrame(summary).set_index('model').style.format('{:.4f}'))
    per_class_df=pd.DataFrame(per_class_rows)
    per_class_summary=per_class_df.groupby(['model','class'])[['precision','recall','f1','support']].mean().reset_index()
    display(per_class_summary.style.format({'precision':'{:.4f}','recall':'{:.4f}','f1':'{:.4f}','support':'{:.1f}'}))
    per_class_df.to_csv(OUT_DIR/'multiclass_test_per_class_by_seed.csv',index=False)
    per_class_summary.to_csv(OUT_DIR/'multiclass_test_per_class_mean.csv',index=False)
    with open(OUT_DIR/'multiclass_results.json','w') as f:
        json.dump({'classes':classes,'cv_oof_smotenc':cv_result,'test_runs':mc_runs,
          'test_distribution':ymc_test.value_counts().to_dict(),'training_distribution':ymc.value_counts().to_dict(),
          'smote_floor':5000,'split':{'train':len(ymc),'test':len(ymc_test)}},f,indent=2)

## 10. Saved results used by the current paper (reference values)
This cell displays the project?s previously generated results from `results/baseline_*.json`, `results/extended.json`, and `results/multiclass_smote.json`?the result files from which the paper tables were prepared. The values are historical saved outputs, **not new Colab execution results**. Use the next cells to rerun the models and compare the fresh output against these references. Small numerical differences can occur when library/hardware versions differ; the paper reports rounded aggregates.

Original paper binary accuracy values are included for context. The cited paper?s reported figures are not expected outputs of this code; the ?reproduced? values are this project?s runs.

In [ ]:
PAPER_REFERENCE = {
  "baseline": {
    "KNN": {
      "accuracy": [
        84.51,
        0.0
      ],
      "FAR": 30.12,
      "AUC": 93.16
    },
    "Naive Bayes": {
      "accuracy": [
        55.62,
        0.0
      ],
      "FAR": 0.01,
      "AUC": 70.12
    },
    "DT": {
      "accuracy": [
        86.37,
        0.05
      ],
      "FAR": 24.95,
      "AUC": 85.59
    },
    "Logistic Regression": {
      "accuracy": [
        80.93,
        0.0
      ],
      "FAR": 39.04,
      "AUC": 95.56
    },
    "AdaBoost": {
      "accuracy": [
        82.57,
        0.0
      ],
      "FAR": 35.9,
      "AUC": 95.91
    },
    "Random Forest": {
      "accuracy": [
        87.06,
        0.06
      ],
      "FAR": 26.97,
      "AUC": 97.85
    },
    "SVM": {
      "accuracy": [
        81.43,
        0.0
      ],
      "FAR": 40.65,
      "AUC": 95.57
    }
  },
  "binary_extension": {
    "RF (baseline, 0.5)": {
      "acc": [
        87.06,
        0.06
      ],
      "f1_macro": [
        86.44,
        0.07
      ],
      "far": 26.97,
      "fnr": 1.49,
      "auc": 97.85,
      "mcc": 75.32
    },
    "RF + balanced": {
      "acc": [
        89.28,
        0.03
      ],
      "f1_macro": [
        88.91,
        0.03
      ],
      "far": 20.85,
      "fnr": 2.46,
      "auc": 98.1,
      "mcc": 79.05
    },
    "RF + balanced + J-thr": {
      "acc": [
        90.67,
        0.07
      ],
      "f1_macro": [
        90.45,
        0.07
      ],
      "far": 16.04,
      "fnr": 3.85,
      "auc": 98.1,
      "mcc": 81.39
    },
    "GBDT (0.5)": {
      "acc": [
        87.55,
        0.05
      ],
      "f1_macro": [
        86.98,
        0.05
      ],
      "far": 25.96,
      "fnr": 1.42,
      "auc": 98.56,
      "mcc": 76.23
    },
    "GBDT + J-thr": {
      "acc": [
        90.77,
        0.05
      ],
      "f1_macro": [
        90.54,
        0.06
      ],
      "far": 16.62,
      "fnr": 3.19,
      "auc": 98.56,
      "mcc": 81.7
    },
    "GBDT + balanced": {
      "acc": [
        90.62,
        0.1
      ],
      "f1_macro": [
        90.36,
        0.1
      ],
      "far": 17.37,
      "fnr": 2.86,
      "auc": 98.58,
      "mcc": 81.47
    },
    "GBDT + balanced + J-thr (proposed)": {
      "acc": [
        90.97,
        0.1
      ],
      "f1_macro": [
        90.74,
        0.1
      ],
      "far": 16.28,
      "fnr": 3.12,
      "auc": 98.58,
      "mcc": 82.08
    }
  },
  "multiclass": {
    "RF baseline (reproduced)": {
      "accuracy": [
        75.43,
        0.08
      ],
      "macro_f1": [
        46.51,
        0.21
      ],
      "per_class": {
        "Analysis": {
          "precision": 0.54,
          "recall": 1.06,
          "f1": 0.72
        },
        "Backdoor": {
          "precision": 1.76,
          "recall": 9.23,
          "f1": 2.96
        },
        "DoS": {
          "precision": 64.16,
          "recall": 9.6,
          "f1": 16.69
        },
        "Exploits": {
          "precision": 61.39,
          "recall": 78.92,
          "f1": 69.06
        },
        "Fuzzers": {
          "precision": 29.59,
          "recall": 58.53,
          "f1": 39.31
        },
        "Generic": {
          "precision": 99.88,
          "recall": 96.78,
          "f1": 98.31
        },
        "Normal": {
          "precision": 96.22,
          "recall": 75.69,
          "f1": 84.73
        },
        "Reconnaissance": {
          "precision": 92.47,
          "recall": 79.9,
          "f1": 85.73
        },
        "Shellcode": {
          "precision": 40.71,
          "recall": 66.83,
          "f1": 50.57
        },
        "Worms": {
          "precision": 47.07,
          "recall": 10.45,
          "f1": 17.08
        }
      }
    },
    "LightGBM + SMOTENC": {
      "accuracy": [
        76.59,
        0.12
      ],
      "macro_f1": [
        51.35,
        0.49
      ],
      "per_class": {
        "Analysis": {
          "precision": 4.9,
          "recall": 12.76,
          "f1": 7.08
        },
        "Backdoor": {
          "precision": 4.43,
          "recall": 13.55,
          "f1": 6.68
        },
        "DoS": {
          "precision": 49.51,
          "recall": 10.72,
          "f1": 17.52
        },
        "Exploits": {
          "precision": 61.47,
          "recall": 84.04,
          "f1": 71.0
        },
        "Fuzzers": {
          "precision": 31.78,
          "recall": 59.53,
          "f1": 41.44
        },
        "Generic": {
          "precision": 99.73,
          "recall": 97.17,
          "f1": 98.44
        },
        "Normal": {
          "precision": 97.01,
          "recall": 75.7,
          "f1": 85.04
        },
        "Reconnaissance": {
          "precision": 93.37,
          "recall": 80.59,
          "f1": 86.51
        },
        "Shellcode": {
          "precision": 30.65,
          "recall": 80.74,
          "f1": 44.43
        },
        "Worms": {
          "precision": 60.08,
          "recall": 51.36,
          "f1": 55.37
        }
      }
    }
  }
}

# Paper-reported binary accuracy values (the original paper is the comparator, not a rerun target).
ORIGINAL_BINARY_ACC = {'KNN':86.03,'Naive Bayes':83.05,'DT':82.27,'SVM':87.21,'Logistic Regression':86.08,'AdaBoost':87.54,'Random Forest':87.82}
rows=[]
for name,ref in PAPER_REFERENCE['baseline'].items():
    rows.append({'model':name,'original_paper_accuracy_%':ORIGINAL_BINARY_ACC.get(name,np.nan),
                 'saved_reproduction_accuracy_%':ref['accuracy'][0], 'reproduction_sd_pp':ref['accuracy'][1],
                 'FAR_%':ref['FAR'],'AUC_%':ref['AUC']})
print('Saved binary baseline reproduction (paper Table 1 comparison):')
display(pd.DataFrame(rows).set_index('model'))
print('Saved binary extension results (paper main ablation table):')
rows=[]
for name,ref in PAPER_REFERENCE['binary_extension'].items():
    rows.append({'configuration':name,'accuracy_mean_%':ref['acc'][0],'accuracy_sd_pp':ref['acc'][1],
      'macro_F1_%':ref['f1_macro'][0],'FAR_%':ref['far'],'FNR_%':ref['fnr'],'AUC_%':ref['auc'],'MCC_%':ref['mcc']})
display(pd.DataFrame(rows).set_index('configuration'))
print('Saved multi-class results from the paper output artifacts (test never resampled):')
rows=[]
for name,ref in PAPER_REFERENCE['multiclass'].items():
    rows.append({'model':name,'accuracy_mean_%':ref['accuracy'][0],'accuracy_sd_pp':ref['accuracy'][1],
      'macro_F1_mean_%':ref['macro_f1'][0],'macro_F1_sd_pp':ref['macro_f1'][1]})
display(pd.DataFrame(rows).set_index('model'))
perclass=[]
for name,ref in PAPER_REFERENCE['multiclass'].items():
    for cls,m in ref['per_class'].items(): perclass.append({'model':name,'class':cls,**{k+'_percent':v for k,v in m.items()}})
display(pd.DataFrame(perclass).set_index(['model','class']))

## 11. Compare newly measured results with the saved reference
This summarizes measured runs only. The prior-paper/project results are shown separately above. Per-class scores are means across final-fit seeds, each evaluated against the unchanged official test rows. When communicating uncertainty, distinguish seed-to-seed standard deviation from a confidence interval. Multi-class model gains should be stated as absolute percentage-point differences in macro-F1 and accuracy.

In [ ]:
if RUN_MULTICLASS:
    comparison=pd.DataFrame(summary).set_index('model')
    comparison['accuracy_gain_vs_RF_pp']=100*(comparison.accuracy_mean-comparison.loc['RF baseline (reproduced)','accuracy_mean'])
    comparison['macro_f1_gain_vs_RF_pp']=100*(comparison.macro_f1_mean-comparison.loc['RF baseline (reproduced)','macro_f1_mean'])
    display(comparison.style.format('{:.4f}'))
    fresh_vs_saved=[]
    for model_name in comparison.index:
        saved=PAPER_REFERENCE['multiclass'][model_name]
        fresh_vs_saved.append({'model':model_name,
          'fresh_accuracy_%':100*comparison.loc[model_name,'accuracy_mean'],
          'saved_paper_accuracy_%':saved['accuracy'][0],
          'fresh_macro_F1_%':100*comparison.loc[model_name,'macro_f1_mean'],
          'saved_paper_macro_F1_%':saved['macro_f1'][0]})
    print('Fresh rerun compared with the saved measurements used in the paper:')
    display(pd.DataFrame(fresh_vs_saved).set_index('model').style.format('{:.2f}'))
    print('Original test class distribution (unchanged):')
    display(ymc_test.value_counts().rename('test_support').to_frame())
print('Saved artifacts in:',OUT_DIR)
for p in sorted(OUT_DIR.glob('*')): print(p.name, p.stat().st_size, 'bytes')
if RUN_BINARY_BASELINES:
    print('Fresh binary baseline accuracy compared with saved reproduction:')
    baseline_name_map={'Decision Tree':'DT','SVM (40k stratified train subset)':'SVM'}
    fresh=[]
    for model_name,runs in binary_baselines.items():
        key=baseline_name_map.get(model_name,model_name); saved=PAPER_REFERENCE['baseline'][key]
        fresh.append({'model':key,'fresh_accuracy_%':100*np.mean([r['accuracy'] for r in runs]),'saved_accuracy_%':saved['accuracy'][0]})
    display(pd.DataFrame(fresh).set_index('model').style.format('{:.2f}'))
if RUN_BINARY_EXTENSION:
    print('Fresh binary extension compared with saved paper results:')
    fresh=[]
    for model_name,runs in variants.items():
        key=model_name.replace('J threshold','J-thr').replace(' (proposed)',' (proposed)')
        key={'RF baseline (0.5)':'RF (baseline, 0.5)'}.get(key,key)
        if key in PAPER_REFERENCE['binary_extension']:
            fresh.append({'configuration':key,'fresh_accuracy_%':100*np.mean([r['accuracy'] for r in runs]),'saved_accuracy_%':PAPER_REFERENCE['binary_extension'][key]['acc'][0],
              'fresh_macro_F1_%':100*np.mean([r['macro_f1'] for r in runs]),'saved_macro_F1_%':PAPER_REFERENCE['binary_extension'][key]['f1_macro'][0]})
    display(pd.DataFrame(fresh).set_index('configuration').style.format('{:.2f}'))


## 12. Updated scenario diagnostics: split shift, operating points, and false alarms

This optional section incorporates the latest `explore3.py` and `fp_analysis.py` diagnostics. It measures feature-vector overlap and label conflict, train/test distribution differences, and where the two binary detectors produce false alarms. A train-vs-test domain classifier uses features only; its grouped folds keep identical feature rows together. Threshold alternatives (Youden J, equal-error rate, and 5%/10% OOF false-alarm targets) are selected from grouped out-of-fold training scores only. The official test labels are used only to report the resulting held-out metrics. These are sensitivity analyses; do not use their test results to choose a model or threshold.

Set `RUN_SHIFT_DIAGNOSTICS=True` to run. The domain classifier adds five fits. This section requires the binary extension cell to have run because it reuses its training-only category encoding and grouped folds.


In [ ]:
RUN_SHIFT_DIAGNOSTICS = False
if RUN_SHIFT_DIAGNOSTICS and RUN_BINARY_EXTENSION:
    from sklearn.metrics import roc_auc_score, roc_curve
    from sklearn.model_selection import StratifiedGroupKFold

    # Duplicate and label-conflict audit on feature rows, excluding IDs and targets.
    h_train = pd.util.hash_pandas_object(Xtr, index=False).to_numpy()
    h_test = pd.util.hash_pandas_object(Xte, index=False).to_numpy()
    h_all = np.concatenate([h_train, h_test])
    y_all = np.concatenate([ytr, yte])
    label_counts = pd.DataFrame({'hash': h_all, 'label': y_all}).groupby('hash').label.nunique()
    audit = {
        'unique_train_rows': int(np.unique(h_train).size),
        'unique_test_rows': int(np.unique(h_test).size),
        'test_feature_rows_also_in_train': int(np.isin(h_test, h_train).sum()),
        'feature_vectors_with_conflicting_labels': int((label_counts > 1).sum()),
        'train_attack_prior_percent': float(100 * ytr.mean()),
        'test_attack_prior_percent': float(100 * yte.mean()),
    }
    print('Feature overlap / prior audit:', audit)

    # Label-free domain classifier: can the feature values identify the official partition?
    # Group identical vectors across both partitions so duplicates cannot cross domain folds.
    X_domain = pd.concat([XA, XB], ignore_index=True)
    y_domain = np.r_[np.zeros(len(XA), dtype=int), np.ones(len(XB), dtype=int)]
    domain_groups = np.r_[h_train, h_test]
    domain_oof = np.zeros(len(X_domain), dtype=float)
    domain_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
    for fold, (fit_i, val_i) in enumerate(domain_cv.split(X_domain, y_domain, groups=domain_groups), 1):
        domain_model = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05,
            num_leaves=31, min_child_samples=100, n_jobs=2, verbosity=-1, random_state=0)
        domain_model.fit(X_domain.iloc[fit_i], y_domain[fit_i], categorical_feature=CAT)
        domain_oof[val_i] = domain_model.predict_proba(X_domain.iloc[val_i])[:, 1]
        print('Domain fold', fold, 'complete')
    domain_auc = float(roc_auc_score(y_domain, domain_oof))
    print('Grouped out-of-fold train-vs-test domain AUC (0.5 is hard to distinguish):', round(domain_auc, 4))

    # Post-hoc operating-point sensitivity. Every threshold comes from training OOF scores.
    fpr_oof, tpr_oof, thr_oof = roc_curve(ytr, oof_bal)
    eer_i = int(np.argmin(np.abs(fpr_oof - (1 - tpr_oof))))
    scenario_thresholds = {'Youden J': float(threshold_bal), 'Equal error rate': float(thr_oof[eer_i])}
    for target_fpr in (0.05, 0.10):
        valid = np.flatnonzero(fpr_oof <= target_fpr)
        scenario_thresholds[f'OOF FAR at most {target_fpr:.0%}'] = float(thr_oof[valid[-1]])
    scenario_rows = []
    for scenario, threshold in scenario_thresholds.items():
        result = binary_metrics(yte, seed0_scores['proposed'] >= threshold, seed0_scores['proposed'])
        scenario_rows.append({'scenario': scenario, 'OOF_threshold': threshold,
            'test_accuracy_%': 100 * result['accuracy'], 'test_FAR_%': 100 * result['FAR'],
            'test_FNR_%': 100 * result['FNR'], 'test_macro_F1_%': 100 * result['macro_f1'],
            'test_AUC_%': 100 * result['AUC']})
    scenario_table = pd.DataFrame(scenario_rows).set_index('scenario')
    display(scenario_table.style.format('{:.3f}'))

    # Locate false alarms among normal test traffic for RF@0.5 and proposed GBDT@Youden.
    normal = yte == 0
    false_alarm_rows = []
    for model_name, pred in [('RF baseline', seed0_scores['rf'] >= 0.5),
                             ('Proposed GBDT', seed0_scores['proposed'] >= threshold_bal)]:
        normal_frame = pd.DataFrame({
            'service': Xte.loc[normal, 'service'].astype(str).to_numpy(),
            'proto_state': (Xte.loc[normal, 'proto'].astype(str) + '/' +
                            Xte.loc[normal, 'state'].astype(str)).to_numpy(),
            'false_alarm': pred[normal].astype(int)})
        for field in ('service', 'proto_state'):
            grouped = normal_frame.groupby(field).false_alarm.agg(['sum', 'count'])
            grouped = grouped[grouped['count'] >= 20].sort_values('sum', ascending=False).head(10)
            for key, row in grouped.iterrows():
                false_alarm_rows.append({'model': model_name, 'group_by': field, 'group': key,
                    'false_alarms': int(row['sum']), 'normal_flows': int(row['count']),
                    'FAR_%': 100 * row['sum'] / row['count']})
    false_alarm_table = pd.DataFrame(false_alarm_rows)
    display(false_alarm_table)
    with open(OUT_DIR/'shift_scenario_diagnostics.json', 'w') as f:
        json.dump({'feature_audit': audit, 'domain_auc_grouped_oof': domain_auc,
            'threshold_scenarios': scenario_rows,
            'false_alarm_groups': false_alarm_rows}, f, indent=2)
    print('Saved shift and scenario diagnostics to', OUT_DIR/'shift_scenario_diagnostics.json')
elif RUN_SHIFT_DIAGNOSTICS:
    print('Enable RUN_BINARY_EXTENSION and run that section first; diagnostics need its OOF scores and fitted predictions.')


## 13. Interpretation, limitations, and reproducibility checklist
Before using results in the paper, verify: (1) both CSV filenames and 175,341/82,332 row counts match the official split; (2) all expected seeds completed; (3) no test rows entered preprocessing fit or a sampler; (4) `attack_cat` has ten classes including `Normal`; (5) per-class precision, recall, F1, macro-F1 and accuracy are reported; (6) baseline and extension use exactly the same test rows; and (7) cited-paper scores are distinguished from this project's measured reproduction.

The original article's undocumented preprocessing and tuning mean exact numerical reproduction cannot be guaranteed. Report the code choices and library versions. The result archive above preserves the measurements used in the current paper; a Colab rerun is a reproduction attempt, not a way to force results to match. The multi-class extension addresses class imbalance using fold-local categorical-aware SMOTENC. Its stratified folds do not group duplicate feature rows, matching the paper. The notebook does not establish improvement until the measured run supports it.

**Output files:** result JSON, per-class CSV files, and figures are stored in the temporary Colab folder `/content/UNSW_NB15_colab_results/`. They are not saved to Drive; download the ZIP in the final section before the runtime ends.

## 14. Package and download results (local only)

This creates a ZIP from the Colab runtime output folder and starts a browser download. No output is copied to Google Drive. Download the archive before the Colab session ends; Colab's local files are temporary.


In [ ]:
DOWNLOAD_RESULTS = True
if DOWNLOAD_RESULTS:
    archive_path = Path('/content/UNSW_NB15_colab_results.zip')
    shutil.make_archive(str(archive_path.with_suffix('')), 'zip', root_dir=str(OUT_DIR))
    print('Created local archive:', archive_path, '| bytes:', archive_path.stat().st_size)
    from google.colab import files
    files.download(str(archive_path))
